# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant schema:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant.Dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns.

**Note:** All references to entities (record sets, fields, columns) use their `@id` values for unambiguous identification.

In [ ]:
# List all record sets in the dataset using their @id values
record_sets = list(dataset.record_sets())
print('Available record sets (by @id):')
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '<missing>')}")

# Explore fields and columns within the first record set
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    fields = dataset.fields(record_set=first_record_set_id)
    print(f"\nFields in record set '{first_record_set_id}':")
    for f in fields:
        print(f"  - @id: {f['@id']}, name: {f.get('name', '<missing>')}, dataType: {f.get('dataType', '<unknown>')}")

    columns = dataset.columns(record_set=first_record_set_id)
    print(f"\nColumns in record set '{first_record_set_id}':")
    for c in columns:
        print(f"  - @id: {c['@id']}, name: {c.get('name', '<missing>')}, source: {c.get('source', '<unknown>')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Create a dictionary of DataFrames, keyed by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show column names from the first DataFrame
if record_set_ids:
    print(f"Columns from record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())

    # Display first rows
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations include:
- Removing outliers
- Normalizing numeric fields
- Grouping data by key attributes

All fields referenced by their `@id`.

In [ ]:
# For demonstration, select the first record set and a numeric field
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else None

if df is not None:
    # List numeric fields by @id
    numeric_fields = [f['@id'] for f in dataset.fields(record_set=record_set_id) 
                     if f.get('dataType', '').lower() in ['integer', 'float', 'number']]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Filtering (e.g., values > threshold)
        threshold = 10
        if numeric_field_id in df.columns:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by another field (e.g., if 'ward' or similar exists)
            group_fields = [f['@id'] for f in dataset.fields(record_set=record_set_id)
                            if f.get('dataType', '').lower() == 'text']
            group_field_id = group_fields[0] if group_fields else None

            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean()
                print(f"Grouped data by {group_field_id}:")
                display(grouped_df.head())
    else:
        print(f"No numeric fields found for record set {record_set_id}.")
else:
    print("No data extracted for analysis.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

*Example: Histogram or barplot of a numeric field, grouped by a categorical field.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution, grouped by the group field, if available
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    if group_field_id and group_field_id in df.columns:
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
    else:
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains ordered logistic regression outputs on adoption predictors for indigenous and modern knowledge in rangeland management.
- Key numeric fields and their distributions were explored and visualized.
- Data may contain missing values and biases; refer to the dataset metadata for limitations and social impact considerations.
- Further analysis can focus on model coefficients, categorical factors of adoption, or fairness aspects described in metadata.

> **Next Steps:**
- You may extend this notebook with more detailed statistical analysis or machine learning tasks using the processed data.
- Review metadata and original documentation for responsible use and ethical considerations.